# Objetivo específico 5: precisión de la adaptación y usabilidad técnica

**Tesis:** *Validación de una arquitectura multiagente basada en inteligencia de enjambre para la adaptación multimodal en Programación.* Versión auditada: informe **D2.9.7-CIERRE-BRECHAS-CORRECCION-CUADROS** (sha256 `a6bb2934de7fd512…`).

## A. Identificación
**OE5 (literal de la tesis):** Determinar el nivel de precisión de la adaptación multimodal y el nivel de usabilidad técnica del artefacto desarrollado mediante la evaluación de expertos y docentes del área de programación, contrastando la puntuación de usabilidad obtenida mediante la escala SUS con el valor de referencia establecido de 75 puntos.

**Meta de desempeño asociada:** Precisión F1_adapt ≥ 0,85 y usabilidad percibida SUS > 75,0/100.

**Pregunta experimental.** ¿Cuál es el nivel de precisión de la adaptación (F1_adapt) y de usabilidad (SUS) frente a las metas F1_adapt ≥ 0,85 y SUS > 75?

**Hipótesis y contraste.** H1 (general): F1_adapt ≥ 0,85 y puntaje SUS > 75,0 (media de al menos 10 evaluadores). El F1_adapt se contrasta con el IC95 % y una prueba unilateral; el SUS con la media de ≥ 10 evaluadores.

**Tablas del informe que se concilian aquí:** 27, 28, 29, 30, 35 y 41. **Campañas:** N5 (F1 v3 provisional) y K10 histórico (v2). Todas son **experimentales y no oficiales**.

**Datos que usa este notebook**
- oe_n5_2026-10-06/f1_v3_provisional/replica_*.json (10 réplicas × 100 perfiles = 1 000 casos; especificación v3 PROVISIONAL, official=False)
- backend/official_runs/k10_official_2026-09-27/replica_*.json (K10 con la especificación v2, histórico)

**Scripts**
- oe_n8_2026-10-13/scripts/reproduce_f1_v3_independent.py (F1, precisión y recall por modalidad)
- oe_n8_2026-10-13/scripts/alineacion_aptitud_f1.py (Tabla 29)
- scripts/reproduce_results.py (Tablas 30 y parte de la 27)

**Limitaciones**
- **No existen datos de usabilidad ni del panel de expertos: 0 evaluadores reales.** La parte SUS del OE5 **no se puede evaluar** y no se simula.
- La especificación v3 del F1 es **provisional**; el resultado con v2 es histórico y corresponde a otra definición de la métrica. La matriz *gold* por arquetipo está pendiente de validación por el panel.
- La evaluación usa perfiles **sintéticos** (4 arquetipos × 5 dificultades × réplicas), no estudiantes reales.
- Las estadísticas por perfil de la Tabla 27 (mediana, desviación, rango) no se recalculan en esta entrega.

> Reproducir una cifra **no** demuestra que una hipótesis sea cierta, ni que los experimentos se hicieran en las condiciones declaradas, ni que los registros sean auténticos: solo que, con estos archivos y estos scripts, el cálculo da el valor del informe. Este notebook es **independiente** de los demás: resuelve sus propias dependencias.

## B. Entorno
Se instalan **solo** las dependencias fijadas (las versiones que cita el informe). No se necesitan tokens, claves ni archivos privados.

In [ ]:
%pip install -q numpy==2.5.4 pandas==3.0.6 scipy==1.18.1 statsmodels==0.15.0 matplotlib==3.11.2
import sys, platform, subprocess
# Los scripts corren en procesos nuevos: estas son las versiones que realmente usan (el kernel de Colab puede traer otras cargadas, p. ej. numpy).
print(subprocess.run([sys.executable, "-c", "import platform, numpy, pandas, scipy, statsmodels, matplotlib as m; "
      "print('Python      ', platform.python_version()); "
      "[print(f'{n:12s}', mod.__version__) for n, mod in (('numpy', numpy), ('pandas', pandas), ('scipy', scipy), ('statsmodels', statsmodels), ('matplotlib', m))]"],
      capture_output=True, text=True).stdout)
print("Sistema:", platform.platform())
# Nota: pip puede avisar de conflictos con paquetes preinstalados de Colab (google-colab, numba); no afectan a este análisis.

## C. Fuentes de datos
El notebook clona el repositorio público en `REF` (clon parcial: `reproducibility/`, el código del proyecto necesario y los datos históricos rastreados en Git). Si ya tiene la carpeta `reproducibility/` se usa esa. Se calcula el SHA-256 de **cada archivo del manifiesto** y se **detiene** el análisis si alguno no coincide: un hash prueba la identidad del archivo frente a una referencia, no que su contenido sea verdadero. Los archivos sanitizados (rutas locales sustituidas) están identificados en `data/metadata/SANITIZACION.csv` y **no** son idénticos a los originales.

In [ ]:
import os, subprocess, shutil, json
from pathlib import Path
import pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 80); pd.set_option("display.max_rows", 60)

REPO_URL = "https://github.com/Renato5Lara/multiagent-llm-education.git"
REF = "repro/auditoria-integral-2026-10-19"   # rama o etiqueta con el paquete

def localizar():
    env = os.environ.get("REPRO_LOCAL_ROOT")
    if env and (Path(env) / "EVIDENCE_MANIFEST.csv").exists():
        return Path(env), f"local (REPRO_LOCAL_ROOT={env})"
    for p in (Path.cwd(), Path.cwd() / "reproducibility", Path.cwd().parent):
        if (p / "EVIDENCE_MANIFEST.csv").exists():
            return p, f"local ({p})"
    dest = Path("/content/tesis_repo") if Path("/content").exists() else Path.cwd() / "tesis_repo"
    if dest.exists():
        shutil.rmtree(dest)
    r = subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse", "--branch", REF, REPO_URL, str(dest)], capture_output=True, text=True)
    if r.returncode == 0:
        r = subprocess.run(["git", "-C", str(dest), "sparse-checkout", "set", "reproducibility", "backend/adaptation_swarm", "backend/official_runs", "backend/loadtest"], capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError(f"No se pudo clonar {REPO_URL} en {REF!r}.\n{r.stderr.strip()}\nCompruebe que la rama/etiqueta existe o copie reproducibility/ a mano.")
    return dest / "reproducibility", f"clonado de {REPO_URL} @ {REF}"

ROOT, origen = localizar()
commit = subprocess.run(["git", "-C", str(ROOT), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip() or "(sin git)"
print("Origen:", origen); print("Raíz  :", ROOT); print("Commit:", commit)

def run_script(name, *args):
    r = subprocess.run([sys.executable, str(ROOT / "scripts" / name), *args], cwd=ROOT / "scripts", capture_output=True, text=True, encoding="utf-8", errors="replace")
    print(r.stdout[-3500:])
    if r.stderr.strip():
        print("[stderr]", r.stderr[-1500:])
    return r.returncode

rc = run_script("verify_hashes.py")
rep_h = json.loads((ROOT / "outputs" / "hash_report.json").read_text(encoding="utf-8"))
if rc != 0:
    raise RuntimeError("ERROR DE INTEGRIDAD: hay archivos alterados o ausentes. Se detiene el análisis (ver outputs/hash_report.json).")
print("Integridad OK:", rep_h["manifiesto"]["coinciden"], "de", rep_h["manifiesto"]["entradas"], "archivos del manifiesto coinciden.")

## D. Preparación y validación de los datos
Se leen los archivos reales, se comprueban esquema, tipos, duplicados, valores ausentes (distintos de cero), unidades y recuentos, y se declara cada exclusión con su regla. **No se elimina ninguna fila.**

In [ ]:
import glob, numpy as np
fs = sorted(glob.glob(str(ROOT / "data/experiments/oe_n5_2026-10-06/f1_v3_provisional/replica_*.json")))
reps = [json.load(open(f, encoding="utf-8")) for f in fs]
casos = [c for r in reps for c in r["cases"]]
print("Réplicas:", len(reps), "| casos:", len(casos), "| perfiles distintos:", len({c["profile_id"] for c in casos}))
f1 = np.array([c["f1"] for c in casos])
print("F1 por caso: finito:", bool(np.isfinite(f1).all()), "| rango", f1.min(), "a", f1.max(), "| valores ausentes: 0")
print("Especificación:", json.load(open(ROOT / "data/experiments/oe_n5_2026-10-06/f1_v3_summary.json", encoding="utf-8"))["definition"])
print("Evaluadores humanos con datos reales (SUS / panel):", 0, "— no hay datos y no se simulan")

## E. Reproducción del análisis
`reproduce_results.py --oe oe5` comprueba, para este objetivo: (1) cuando existen, que cada fila de `runs.csv` se recalcula con el código del proyecto desde los **registros por petición**; (2) que los **scripts de análisis del proyecto**, ejecutados **sin modificar** sobre una copia en la que se borraron antes sus salidas, regeneran lo publicado; y (3) que las **cifras del informe** coinciden con lo recalculado a su precisión publicada. Tolerancias: media unidad del último decimal publicado para las cifras del informe; para comparar archivos regenerados, igualdad byte a byte o salvo finales de línea (los publicados son de Windows) o con ruido de punto flotante (relativa 1e-6, absoluta 1e-12).

In [ ]:
rc = run_script("reproduce_results.py", "--oe", "oe5")
ver = json.loads((ROOT / "outputs" / "verification_report_oe5.json").read_text(encoding="utf-8"))
print("Entorno de los scripts:", ver["entorno"])
print("\nEslabón 1 — peticiones → corrida:")
display(pd.DataFrame(ver["cadena_peticiones_a_corridas"]) if ver["cadena_peticiones_a_corridas"] else "no aplica a este objetivo (sin runs.csv HTTP; los datos son observaciones por ciclo)")
b = pd.DataFrame(ver["derivados_byte_a_byte"])
print("\nEslabón 2 — derivados regenerados vs publicados:", b.estado.value_counts().to_dict() if len(b) else "no aplica")

## F. Resultados y comparación con el informe
Tabla principal de este objetivo: **Tabla 28**. Cada fila compara el valor **publicado** (transcrito de `data/metadata/informe_D2.9.7_tablas.json`, extraído del DOCX con su SHA-256) con el **recalculado**, con la celda exacta del informe.

In [ ]:
comp = pd.DataFrame(ver["comprobaciones"])
comp["tabla_n"] = comp["tabla"].str.extract(r"(\d+)")[0]
cols = ["etiqueta_fila", "encabezado", "publicado", "recalculado", "dif_abs", "tolerancia", "estado", "estado_cifra"]
for tb, g in comp.groupby("tabla"):
    print(f"\n=== {tb}: {len(g)} cifras ===")
    display(g[g.estado != "NO_REPRODUCIBLE"][[c for c in cols if c in g]].head(12))
print("\nCifras que NO se pueden reproducir y por qué:")
nr_df = comp[comp.estado == "NO_REPRODUCIBLE"].reindex(columns=["tabla", "id", "motivo"])
print("(ninguna)" if nr_df.empty else "")
display(nr_df)

## G. Verificación
Resumen de estados por cifra, **prueba negativa** del comparador (una cifra alterada a propósito debe detectarse) y pruebas unitarias rápidas.

In [ ]:
cnt = ver["conteo_estado_cifra"]
print("Cifras por estado:", cnt)
print("Detalle de comparación:", ver["conteo_estados"])
# --- prueba negativa: alteramos una copia en memoria de una cifra publicada; el comparador debe marcar DISCREPANCIA
sys.path.insert(0, str(ROOT / "scripts"))
from repro_lib import compare
ok = comp[(comp.estado == "COINCIDE") & comp.recalculado.notna()].iloc[0]
alterada = compare(str(ok["publicado"]), float(ok["recalculado"]) * 1.5 + 1)
print("\nPrueba negativa sobre", ok["id"], "(publicado", ok["publicado"], "; se compara contra un valor deliberadamente distinto):", alterada["estado"])
assert alterada["estado"] in ("DISCREPANCIA", "DIFERENCIA_SIN_EXPLICAR"), "el comparador NO detectó una diferencia real"
# --- pruebas unitarias rápidas del comparador y del clasificador de redondeo
%pip install -q pytest
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", str(ROOT / "tests" / "test_reproducibility.py"), "-k", "comparar_archivos or clasificacion or parse or sanitizadas or rutas_locales"], capture_output=True, text=True, cwd=ROOT)
print(r.stdout[-600:])
nr = cnt.get("NO_REPRODUCIBLE", 0); disc = cnt.get("DISCREPANCIA", 0)
assert disc == 0, "hay discrepancias reales: revise la tabla de comparación"
print("\nSin discrepancias." if disc == 0 else "")

## H. Conclusión limitada a la evidencia
Los datos reproducen el F1_adapt con v3 provisional (≈ 0,816; el límite superior del IC95 % queda **por debajo** de 0,85) y el valor histórico con v2 (≈ 0,641). Por tanto **la meta F1_adapt ≥ 0,85 no se alcanza con los datos disponibles**, y el componente de usabilidad (SUS > 75) **no puede evaluarse**: el OE5 solo está evidenciado parcialmente.

**Lo que este notebook no demuestra:** causalidad; autenticidad histórica de las mediciones (los hashes solo prueban identidad frente a una referencia); validación independiente de terceros (el re-análisis lo realizó el propio proyecto); desempeño en condiciones objetivo; evaluación por usuarios o expertos reales.

## I. Exportación y resumen
Se guardan el informe y las tablas de este objetivo en una carpeta **nueva** (los archivos originales no se sobrescriben) y se imprime una línea de resumen para el registro de ejecución.

In [ ]:
import datetime, zipfile
dest = Path("verificacion_oe5_" + datetime.datetime.now().strftime("%Y%m%d_%H%M%S"))
dest.mkdir()
for f in [ROOT / "outputs" / f"verification_report_oe5.json", ROOT / "outputs" / "hash_report.json"] + list((ROOT / "outputs" / "tables").glob("*_oe5.csv")):
    shutil.copy2(f, dest / f.name)
zpath = shutil.make_archive(str(dest), "zip", dest)
print("Exportado en:", dest.resolve(), "\nZIP:", zpath)
try:
    from google.colab import files
    files.download(zpath)
except ImportError:
    pass
resumen = {"objetivo": "OE5", "commit": commit[:8], "cifras": len(ver["comprobaciones"]), "estado_cifra": cnt, "discrepancias": disc, "no_reproducibles": nr,
           "cadena": {c["campana"]: c["estado"] for c in ver["cadena_peticiones_a_corridas"]}, "derivados": b.estado.value_counts().to_dict() if len(b) else {}}
print("RESUMEN_COLAB", json.dumps(resumen, ensure_ascii=False))